# Neural A\* — Fast Track to a Working Bitfile

**One notebook: tiny model → trained → exported → synthesised → deployable.**
Target is a `.bit` file that runs on the board, not a good model.

### Why the last build failed, in one line

The encoder was 391k parameters and 399 MMAC for a 32×32 maze. At the
parallelism needed to be merely *as fast as the CPU*, it wanted **184,839 LUTs
against the Z-7020's 53,200** — 3.47× over. It was never going to fit.

### Why this one will

| encoder | params | MACs | ms @ 100 MHz | est. LUT | of Z-7020 |
|---|---|---|---|---|---|
| original | 389,952 | 399 M | 11.8 | 194,680 | **366%** |
| **tiny 8/16/16** | **3,744** | **3.83 M** | **1.47** | ~28,400 | **53%** |

104× fewer parameters. The measured CPU baseline is 6.70 ms, so this is
**4.5× faster than the CPU and uses half the chip**. The oversized encoder was
costing unnecessary accuracy and unaffordable throughput.

It is trained by distillation — the big float encoder produces the target
guidance maps and the tiny one learns to copy them. No differentiable A\*
needed, and it takes minutes.

### Timing

| section | time |
|---|---|
| 1–3 train + evaluate | ~10 min on GPU |
| 4 export | seconds |
| 5–6 folding + estimate build | ~5 min |
| 7 full build | 30–60 min |

Every toolchain trap encountered is already handled here: the estimate flow stops
before HLS, nodes get named before folding is applied, folding is matched by
layer dimensions, and the constraints are checked before anything long runs.

## 0. Setup

In [2]:
import os, sys, json, glob, time, shutil, subprocess
import numpy as np
import torch
import torch.nn as nn

# WORKDIR = os.path.expanduser("~/fpga_hw2")
WORKDIR = os.path.expanduser("~/Project_2026_NeuralA_Star_New/neural_astar_project")
os.chdir(WORKDIR)

import brevitas.nn as qnn
from brevitas.quant import Int8ActPerTensorFloat
from brevitas.export import export_qonnx

from neural_astar_core import (sample_problems, load_encoder_from_checkpoint,
                               VanillaAStarPlanner, NeuralAStarPlanner, evaluate)
from neural_astar_core.grid import index_to_one_hot
from neural_astar_core.encoder import encoder_input

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA = "mazes_032_moore_c8.npz"
CKPT = "model/mazes_032_moore_c8/lightning_logs/"
PYNQ_Z2 = {"LUT": 53200, "FF": 106400, "BRAM_36K": 140, "DSP": 220}
CLK_NS = 10.0

assert os.path.exists(DATA), "run fetch_data.py first"
print(f"device {DEVICE} | torch {torch.__version__}")

device cuda | torch 2.5.1+cu121


## 1. Architecture and folding

`plan_folding` allocates parallelism greedily to whichever layer is currently
the bottleneck, respecting all three hardware constraints: `SIMD` divides the
layer's input channels, `PE` divides its output channels, and `SIMD ≥ MW/1024`
(an HLS limit that only surfaces at step 11 of the build).

`MAC_BUDGET` is the one knob. Roughly 540 LUTs per MAC/cycle plus ~10k fixed
overhead, measured from the failed build — so 34 lands near 28k LUTs, about
half the device. Raise it for speed, lower it if the estimate comes back tight.

In [3]:
CHANNELS = (8, 16, 16)     # 2 -> 8 -> 16 -> 16 -> 1
WEIGHT_BITS, ACT_BITS = 4, 8
MAC_BUDGET = 34            # ~28k LUTs, ~53% of a Z-7020


def plan_folding(channels, in_dim=2, budget=MAC_BUDGET, pixels=1024):
    widths = [in_dim] + list(channels) + [1]
    dims = [(widths[i] * 9, widths[i + 1]) for i in range(len(widths) - 1)]
    ins = [widths[i] for i in range(len(widths) - 1)]
    outs = [widths[i + 1] for i in range(len(widths) - 1)]
    fold = [[1, 1] for _ in dims]

    def cycles(i):
        mw, mh = dims[i]
        return (mw // fold[i][0]) * (mh // fold[i][1]) * pixels

    def used():
        return sum(s * p for s, p in fold)

    while True:
        i = max(range(len(dims)), key=cycles)
        mw, mh = dims[i]
        s, p = fold[i]
        chosen = None
        for ns, np_ in ((s * 2, p), (s, p * 2)):
            legal = (ns <= ins[i] and ins[i] % ns == 0 and mw % ns == 0
                     and np_ <= outs[i] and outs[i] % np_ == 0
                     and ns >= mw / pixels)
            if legal and used() - s * p + ns * np_ <= budget:
                chosen = (ns, np_)
                break
        if chosen is None:
            break
        fold[i] = list(chosen)
    return dims, [tuple(f) for f in fold]


DIMS, FOLD = plan_folding(CHANNELS)
mac_cycle = sum(s * p for s, p in FOLD)
bottleneck = max((mw // s) * (mh // p) * 1024 for (mw, mh), (s, p) in zip(DIMS, FOLD))
est_lut = 10000 + 540 * mac_cycle

print(f"{'conv':6} {'MW':>5} {'MH':>4} {'SIMD':>5} {'PE':>4} {'MAC/cy':>7} {'cycles':>10}")
print("-" * 46)
for i, ((mw, mh), (s, p)) in enumerate(zip(DIMS, FOLD)):
    print(f"conv{i}  {mw:5d} {mh:4d} {s:5d} {p:4d} {s*p:7d} "
          f"{(mw//s)*(mh//p)*1024:10,d}")
print("-" * 46)
print(f"parallelism : {mac_cycle} MACs/cycle")
print(f"throughput  : {bottleneck*CLK_NS/1e6:.2f} ms/frame at {1000/CLK_NS:.0f} MHz")
print(f"estimated   : {est_lut:,.0f} LUTs = {100*est_lut/PYNQ_Z2['LUT']:.0f}% of the device")
print(f"CPU baseline: 6.70 ms  ->  speedup {6.70/(bottleneck*CLK_NS/1e6):.1f}x")

conv      MW   MH  SIMD   PE  MAC/cy     cycles
----------------------------------------------
conv0     18    8     2    1       2     73,728
conv1     72   16     8    1       8    147,456
conv2    144   16    16    1      16    147,456
conv3    144    1     1    1       1    147,456
----------------------------------------------
parallelism : 27 MACs/cycle
throughput  : 1.47 ms/frame at 100 MHz
estimated   : 24,580 LUTs = 46% of the device
CPU baseline: 6.70 ms  ->  speedup 4.5x


## 2. Build and train the tiny encoder

Distillation: the trained 391k float encoder produces target guidance maps, and
the tiny quantised model learns to reproduce them. It never sees a maze solution
directly — the big model already encodes what a good guidance map looks like.

In [4]:
class QuantGuidanceEncoder(nn.Module):
    """Quantised guidance-map encoder at arbitrary width.

    bias=False everywhere: each conv is followed by BatchNorm, whose shift
    subsumes any bias, and it gives FINN one less thing to fold.
    """

    def __init__(self, weight_bits=4, act_bits=8, in_dim=2, channels=CHANNELS):
        super().__init__()
        self.weight_bits, self.act_bits = weight_bits, act_bits
        self.in_dim, self.channels = in_dim, tuple(channels)
        widths = [in_dim] + list(channels) + [1]

        self.input_q = qnn.QuantIdentity(act_quant=Int8ActPerTensorFloat,
                                         return_quant_tensor=True)
        blocks = []
        for i in range(len(widths) - 1):
            blocks.append(qnn.QuantConv2d(
                widths[i], widths[i + 1], 3, 1, 1, bias=False,
                weight_bit_width=weight_bits, return_quant_tensor=False))
            blocks.append(nn.BatchNorm2d(widths[i + 1]))
            if i != len(widths) - 2:
                blocks.append(qnn.QuantReLU(bit_width=act_bits,
                                            return_quant_tensor=True))
        self.model = nn.Sequential(*blocks)
        self.out_q = qnn.QuantIdentity(bit_width=act_bits,
                                       return_quant_tensor=False)

    def forward(self, x):
        return self.out_q(torch.sigmoid(self.model(self.input_q(x))))


float_encoder = load_encoder_from_checkpoint(CKPT, device=DEVICE)
tiny = QuantGuidanceEncoder(WEIGHT_BITS, ACT_BITS, channels=CHANNELS).to(DEVICE)
print(f"float : {sum(p.numel() for p in float_encoder.parameters()):,} params")
print(f"tiny  : {sum(p.numel() for p in tiny.parameters()):,} params")

float : 391,395 params
tiny  : 3,831 params


In [5]:
N_MAPS, STARTS = 800, 4      # 3200 instances
EPOCHS, BATCH, LR = 120, 64, 3e-3


def distillation_set(n_maps=N_MAPS, starts=STARTS, seed=7):
    ps = sample_problems(DATA, "train", n_maps * starts, seed=seed,
                         starts_per_map=starts, replace=True)
    X = torch.cat([encoder_input(p.obstacle_map,
                                 index_to_one_hot(p.start, p.height, p.width),
                                 index_to_one_hot(p.goal, p.height, p.width))
                   for p in ps], 0)
    float_encoder.eval()
    with torch.no_grad():
        Y = torch.cat([float_encoder(X[i:i+256].to(DEVICE)).cpu()
                       for i in range(0, len(X), 256)], 0)
    return X, Y


t0 = time.time()
X, Y = distillation_set()
print(f"{len(X)} instances in {time.time()-t0:.0f}s")

opt = torch.optim.Adam(tiny.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
loss_fn = nn.L1Loss()

t0 = time.time()
tiny.train()
for ep in range(EPOCHS):
    perm = torch.randperm(len(X))
    total, nb = 0.0, 0
    for i in range(0, len(X), BATCH):
        idx = perm[i:i + BATCH]
        opt.zero_grad()
        loss = loss_fn(tiny(X[idx].to(DEVICE)), Y[idx].to(DEVICE))
        loss.backward(); opt.step()
        total += loss.item(); nb += 1
    sched.step()
    if ep % 20 == 0 or ep == EPOCHS - 1:
        print(f"  epoch {ep+1:4d}/{EPOCHS}  L1 {total/nb:.5f}")
tiny.eval()
print(f"trained in {(time.time()-t0)/60:.1f} min")

3200 instances in 2s


$HOME/miniforge3/envs/fpga_hw2/lib/python3.10/site-packages/torch/_tensor.py:1488: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at ../c10/core/TensorImpl.h:1928.)
  return super().rename(names)


  epoch    1/120  L1 0.16273
  epoch   21/120  L1 0.08376
  epoch   41/120  L1 0.07901
  epoch   61/120  L1 0.07719
  epoch   81/120  L1 0.07528
  epoch  101/120  L1 0.07481
  epoch  120/120  L1 0.07216
trained in 2.8 min


## 3. Is it actually useful?

Accuracy is not the point today, but a model that has learned nothing would
make the whole exercise pointless. `p_exp` is the number to watch: it is the
reduction in node expansions versus plain A\*, and anything meaningfully above
zero means the guidance map is doing real work.

In [6]:
eval_set = sample_problems(DATA, "test", 200, seed=11, starts_per_map=2)
vanilla = VanillaAStarPlanner()

print(f"{'':22} {'p_opt':>7} {'p_exp':>7} {'p_gen':>7}")
results = {}
for name, enc in [(f"float {sum(p.numel() for p in float_encoder.parameters())//1000}k",
                   float_encoder),
                  (f"tiny {sum(p.numel() for p in tiny.parameters())//1000}k W{WEIGHT_BITS}A{ACT_BITS}",
                   tiny)]:
    r = evaluate(eval_set, vanilla, NeuralAStarPlanner(enc, device=DEVICE))
    results[name] = r
    print(f"{name:22} {r.p_opt:7.3f} {r.p_exp:7.3f} {r.p_gen:7.3f}")

tiny_r = list(results.values())[-1]
if tiny_r.p_exp < 0.10:
    print("\nWARNING: p_exp below 0.10 -- the tiny model has barely learned.")
    print("Raise EPOCHS, widen CHANNELS, or increase LR, then re-run section 2.")
else:
    print(f"\nGood enough to build. Expansions cut {tiny_r.p_exp:.0%} vs plain A*.")

torch.save(tiny.state_dict(), f"tiny_encoder_{'_'.join(map(str,CHANNELS))}.pt")

                         p_opt   p_exp   p_gen
float 391k               0.800   0.441   0.239
tiny 3k W4A8             0.750   0.407   0.221

Good enough to build. Expansions cut 41% vs plain A*.


## 4. Export to QONNX

In [7]:
import onnx
from collections import Counter

ONNX = os.path.join(WORKDIR, f"tiny_astar_w{WEIGHT_BITS}a{ACT_BITS}.onnx")
tiny.cpu().eval()
export_qonnx(tiny, input_t=torch.zeros(1, 2, 32, 32), export_path=ONNX)

# FINN requires any BipolarQuant scale to be exactly 1.0 (same fix as HW2).
m = onnx.load(ONNX)
bip = {n.input[1] for n in m.graph.node if n.op_type == "BipolarQuant"}
for init in m.graph.initializer:
    if init.name in bip:
        init.raw_data = np.ones(1, dtype=np.float32).tobytes()
        init.float_data[:] = []
onnx.save(m, ONNX)
tiny.to(DEVICE)

print(f"{os.path.basename(ONNX)}  ({os.path.getsize(ONNX)//1024} KB)")
print("nodes:", dict(Counter(n.op_type for n in m.graph.node)))

np.savez("tiny_test_vectors.npz", x=X[:64].numpy(),
         y=tiny(X[:64].to(DEVICE)).detach().cpu().numpy())
print("saved tiny_test_vectors.npz for board-side verification")

tiny_astar_w4a8.onnx  (24 KB)
nodes: {'Quant': 9, 'Conv': 4, 'BatchNormalization': 4, 'Relu': 3, 'Sigmoid': 1}
saved tiny_test_vectors.npz for board-side verification


## 5. FINN configuration

Everything learned the hard way is encoded here:

* **`steps=` not just `generate_outputs=`.** `generate_outputs` controls what
  FINN *writes*; without an explicit step list it runs all 19 including Vivado,
  so an "estimate-only" build would still take an hour.
* **`step_name_nodes`.** Brevitas exports unnamed nodes and
  `step_specialize_layers` creates more. Unnamed layers make the estimate
  reports collapse to one blank key and the folding config unaddressable.
* **Folding matched by `(MW, MH)`.** Node names cannot be predicted —
  specialisation splits layers between `rtl` and `hls` and numbers each family
  separately.

In [8]:
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.custom_op.registry import getCustomOp
from qonnx.transformation.general import (GiveUniqueNodeNames,
                                          GiveReadableTensorNames)

# Vivado wants libtinfo.so.5; conda ships .so.6. Expose only that symlink
# rather than the whole conda lib/, which floods the log with warnings.
_shim = os.path.expanduser("~/vivado_libs")
os.makedirs(_shim, exist_ok=True)
_t = os.path.join(sys.prefix, "lib", "libtinfo.so.6")
_l = os.path.join(_shim, "libtinfo.so.5")
if os.path.exists(_t) and not os.path.exists(_l):
    os.symlink(_t, _l)
os.environ["LD_LIBRARY_PATH"] = _shim + ":" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["FINN_ROOT"] = os.path.expanduser("~/finn")


def step_name_nodes(model, cfg):
    model = model.transform(GiveUniqueNodeNames())
    model = model.transform(GiveReadableTensorNames())
    return model


def _with_naming(steps):
    steps = list(steps)
    steps.insert(steps.index("step_specialize_layers") + 1, step_name_nodes)
    return steps


def _estimate_steps():
    s = getattr(build_cfg, "estimate_only_dataflow_steps", None)
    if s:
        return list(s)
    full = build_cfg.default_build_dataflow_steps
    return list(full[:full.index("step_generate_estimate_reports") + 1])


ESTIMATE_STEPS = _with_naming(_estimate_steps())
FULL_STEPS = _with_naming(build_cfg.default_build_dataflow_steps)

SPECIALIZE = os.path.join(WORKDIR, "specialize_tiny.json")
json.dump({"Defaults": {"preferred_impl_style": ["rtl", "all"]}},
          open(SPECIALIZE, "w"), indent=2)


def make_cfg(output_dir, build_dir, folding=None, bitfile=False):
    os.environ["FINN_BUILD_DIR"] = build_dir
    os.makedirs(build_dir, exist_ok=True)
    outs = [build_cfg.DataflowOutputType.ESTIMATE_REPORTS]
    if bitfile:
        outs += [build_cfg.DataflowOutputType.BITFILE,
                 build_cfg.DataflowOutputType.PYNQ_DRIVER,
                 build_cfg.DataflowOutputType.DEPLOYMENT_PACKAGE]
    return build_cfg.DataflowBuildConfig(
        output_dir=output_dir, board="Pynq-Z2",
        shell_flow_type=build_cfg.ShellFlowType.VIVADO_ZYNQ,
        synth_clk_period_ns=CLK_NS,
        generate_outputs=outs,
        folding_config_file=folding,
        specialize_layers_config_file=SPECIALIZE,
        auto_fifo_depths=False, verify_save_rtlsim_waveforms=False,
        steps=FULL_STEPS if bitfile else ESTIMATE_STEPS)


def fresh(*dirs):
    for d in dirs:
        p = os.path.join(WORKDIR, d)
        if os.path.exists(p):
            shutil.rmtree(p, ignore_errors=True)


def load_report(output_dir, name):
    p = os.path.join(output_dir, "report", name)
    return json.load(open(p)) if os.path.exists(p) else None


print(f"estimate flow {len(ESTIMATE_STEPS)} steps | full flow {len(FULL_STEPS)} steps")

estimate flow 11 steps | full flow 20 steps


## 6. Estimate build

Minutes, no Vivado. Confirms the design fits before committing to synthesis.

In [9]:
EST = os.path.join(WORKDIR, "tiny_est")
fresh("tiny_est", "tiny_est_tmp")

t0 = time.time()
build.build_dataflow_cfg(ONNX, make_cfg(EST, os.path.join(WORKDIR, "tiny_est_tmp")))
print(f"\nestimate build in {time.time()-t0:.0f}s")

Building dataflow accelerator from $HOME/project/tiny_astar_w4a8.onnx
Intermediate outputs will be generated in $HOME/project/tiny_est_tmp
Final outputs will be generated in $HOME/project/tiny_est
Build log is at $HOME/project/tiny_est/build_dataflow.log
Running step: step_qonnx_to_finn [1/11]
Running step: step_tidy_up [2/11]
Running step: step_streamline [3/11]
Running step: step_convert_to_hw [4/11]
Running step: step_create_dataflow_partition [5/11]
Running step: step_specialize_layers [6/11]
Running step: step_name_nodes [7/11]
Running step: step_target_fps_parallelization [8/11]
Running step: step_apply_folding_config [9/11]
Running step: step_minimize_bit_width [10/11]
Running step: step_generate_estimate_reports [11/11]
Completed successfully

estimate build in 2s


In [10]:
def discover_layers(output_dir):
    # FINN writes intermediate_models/<step>.onnx AFTER each step, so the
    # specialize snapshot predates step_name_nodes. Scan newest-first for one
    # that already has names; otherwise derive them the way
    # GiveUniqueNodeNames does -- {op_type}_{per-type counter}, graph order.
    hits = glob.glob(os.path.join(output_dir, "intermediate_models", "*.onnx"))
    if not hits:
        raise RuntimeError("no intermediate models -- estimate build failed?")
    hits.sort(key=os.path.getmtime)

    def mvaus(path):
        model = ModelWrapper(path)
        out = []
        for n in model.graph.node:
            if "MVAU" not in n.op_type and "VVAU" not in n.op_type:
                continue
            try:
                inst = getCustomOp(n)
                out.append((n.name, n.op_type,
                            inst.get_nodeattr("MW"), inst.get_nodeattr("MH")))
            except Exception:
                out.append((n.name, n.op_type, None, None))
        return out

    for path in reversed(hits):
        got = mvaus(path)
        if got and all(n for n, _, _, _ in got):
            return os.path.basename(path), got
    for path in reversed(hits):
        got = mvaus(path)
        if got:
            ctr, named = {}, []
            for _, op, mw, mh in got:
                i = ctr.get(op, 0); ctr[op] = i + 1
                named.append((f"{op}_{i}", op, mw, mh))
            return os.path.basename(path) + " (names derived)", named
    raise RuntimeError("no MVAU nodes found")


def write_folding(output_dir, path, dims=None, fold=None):
    dims, fold = dims or DIMS, fold or FOLD
    src, layers = discover_layers(output_dir)
    print(f"layers from {src}\n")
    by_dim = {d: f for d, f in zip(dims, fold)}

    cfg = {"Defaults": {}}
    print(f"{'node':<22} {'op_type':<12} {'MW':>5} {'MH':>4} {'SIMD':>5} {'PE':>4}")
    print("-" * 58)
    for i, (name, op, mw, mh) in enumerate(layers):
        f = by_dim.get((mw, mh)) or (fold[i] if i < len(fold) else None)
        if f is None:
            continue
        cfg[name] = {"SIMD": f[0], "PE": f[1]}
        print(f"{name:<22} {op:<12} {str(mw):>5} {str(mh):>4} {f[0]:5d} {f[1]:4d}")

    missing = [n for n, _, _, _ in layers if n not in cfg]
    if missing:
        raise RuntimeError(f"no folding for {missing} -- would run at SIMD=1 "
                           "and fail HLS codegen")
    json.dump(cfg, open(path, "w"), indent=2)
    print(f"\nwrote {os.path.basename(path)} ({len(cfg)-1} layers)")
    return cfg


FOLDING = os.path.join(WORKDIR, "folding_tiny.json")
write_folding(EST, FOLDING)

layers from step_generate_estimate_reports.onnx

node                   op_type         MW   MH  SIMD   PE
----------------------------------------------------------
MVAU_hls_0             MVAU_hls        18    8     2    1
MVAU_hls_1             MVAU_hls        72   16     8    1
MVAU_hls_2             MVAU_hls       144   16    16    1
MVAU_rtl_0             MVAU_rtl       144    1     1    1

wrote folding_tiny.json (4 layers)


{'Defaults': {},
 'MVAU_hls_0': {'SIMD': 2, 'PE': 1},
 'MVAU_hls_1': {'SIMD': 8, 'PE': 1},
 'MVAU_hls_2': {'SIMD': 16, 'PE': 1},
 'MVAU_rtl_0': {'SIMD': 1, 'PE': 1}}

In [11]:
# Re-estimate with folding, and check it fits before the long build.
EST2 = os.path.join(WORKDIR, "tiny_est_folded")
fresh("tiny_est_folded", "tiny_est_folded_tmp")

t0 = time.time()
build.build_dataflow_cfg(ONNX, make_cfg(
    EST2, os.path.join(WORKDIR, "tiny_est_folded_tmp"), folding=FOLDING))
print(f"done in {time.time()-t0:.0f}s\n")

cyc = load_report(EST2, "estimate_layer_cycles.json")
res = load_report(EST2, "estimate_layer_resources.json")

if cyc:
    worst = max(cyc.values())
    for k, v in sorted(cyc.items(), key=server kv: -kv[1])[:8]:
        print(f"  {k:<34} {v:10,d}")
    print(f"\nthroughput : {worst*CLK_NS/1e6:.2f} ms/frame")
    print(f"CPU        : 6.70 ms  ->  {6.70/(worst*CLK_NS/1e6):.1f}x")

if res:
    t = res.get("total", {})
    lut, bram = t.get("LUT", 0), t.get("BRAM_18K", 0)
    print(f"\n{'':10} {'LUT':>9} {'BRAM_18K':>10} {'DSP':>6}")
    print(f"{'estimated':10} {lut:9.0f} {bram:10.0f} {t.get('DSP',0):6.0f}")
    print(f"{'available':10} {PYNQ_Z2['LUT']:9d} {PYNQ_Z2['BRAM_36K']*2:10d} "
          f"{PYNQ_Z2['DSP']:6d}")
    pct = 100 * lut / PYNQ_Z2["LUT"]
    print(f"\nLUT utilisation {pct:.0f}%")
    if pct > 80:
        print("TIGHT -- lower MAC_BUDGET in section 1 and re-run from there.")
        print("Estimates run optimistic; the last build failed at 347%.")
    else:
        print("Comfortable. Proceed to the full build.")

Building dataflow accelerator from $HOME/project/tiny_astar_w4a8.onnx
Intermediate outputs will be generated in $HOME/project/tiny_est_folded_tmp
Final outputs will be generated in $HOME/project/tiny_est_folded
Build log is at $HOME/project/tiny_est_folded/build_dataflow.log
Running step: step_qonnx_to_finn [1/11]
Running step: step_tidy_up [2/11]
Running step: step_streamline [3/11]
Running step: step_convert_to_hw [4/11]
Running step: step_create_dataflow_partition [5/11]
Running step: step_specialize_layers [6/11]
Running step: step_name_nodes [7/11]
Running step: step_target_fps_parallelization [8/11]
Running step: step_apply_folding_config [9/11]
Running step: step_minimize_bit_width [10/11]
Running step: step_generate_estimate_reports [11/11]
Completed successfully
done in 2s

  MVAU_hls_1                            147,456
  MVAU_hls_2                            147,456
  MVAU_rtl_0                            147,456
  MVAU_hls_0                             73,728
  ConvolutionI

## 7. Full build

30–60 minutes for a design this size. **Use a long allocation** — an
interactive `srun` that expires takes the build with it, and tmux does not help
because Slurm kills the whole job step:

```bash
srun --gres=gpu:1 --time=12:00:00 --mem=32G --cpus-per-task=2 --pty bash
```

The GPU is idle during synthesis but usually schedules faster.

Progress does not always reach the notebook — FINN redirects its own stdout.
Watch the filesystem from a second terminal instead:

```bash
watch -n 60 'ls -lt ~/fpga_hw2/tiny_output/intermediate_models/ | head -3'
```

In [12]:
BUILD = os.path.join(WORKDIR, "tiny_output")
fresh("tiny_output", "tiny_build_tmp")

print(f"full build started {time.strftime('%H:%M:%S')} -- 30-60 min\n")
t0 = time.time()
build.build_dataflow_cfg(ONNX, make_cfg(
    BUILD, os.path.join(WORKDIR, "tiny_build_tmp"), folding=FOLDING, bitfile=True))
print(f"\nBUILD COMPLETE in {(time.time()-t0)/60:.0f} min")

full build started 01:48:56 -- 30-60 min

Building dataflow accelerator from $HOME/project/tiny_astar_w4a8.onnx
Intermediate outputs will be generated in $HOME/project/tiny_build_tmp
Final outputs will be generated in $HOME/project/tiny_output
Build log is at $HOME/project/tiny_output/build_dataflow.log
Running step: step_qonnx_to_finn [1/20]
Running step: step_tidy_up [2/20]
Running step: step_streamline [3/20]
Running step: step_convert_to_hw [4/20]
Running step: step_create_dataflow_partition [5/20]
Running step: step_specialize_layers [6/20]
Running step: step_name_nodes [7/20]
Running step: step_target_fps_parallelization [8/20]
Running step: step_apply_folding_config [9/20]
Running step: step_minimize_bit_width [10/20]
Running step: step_generate_estimate_reports [11/20]
Running step: step_hw_codegen [12/20]
Running step: step_hw_ipgen [13/20]
Running step: step_set_fifo_depths [14/20]
Running step: step_create_stitched_ip [15/20]
Running step: step_measure_rtlsim_performance [16

In [13]:
top = (load_report(BUILD, "post_synth_resources.json") or {}).get("(top)", {})
if top:
    print(f"{'Resource':<12} {'Used':>9} {'Available':>11} {'Util':>8}")
    print("-" * 44)
    for key, avail in PYNQ_Z2.items():
        used = top.get(key, top.get(key.replace("_36K", ""), 0))
        print(f"{key:<12} {used:9.0f} {avail:11d} {100*used/avail:7.1f}%")

timing = load_report(BUILD, "post_route_timing.json")
if timing:
    print("\ntiming:", timing)

deploy = os.path.join(BUILD, "deploy")
if os.path.isdir(deploy):
    print(f"\nDEPLOYMENT PACKAGE: {deploy}")
    for f in sorted(glob.glob(deploy + "/**/*", recursive=True))[:12]:
        print("   ", os.path.relpath(f, deploy))
    print("\nCopy to the board:")
    print(f"  scp -r {deploy} xilinx@<board-ip>:~/tiny_astar")
    print("  also copy tiny_test_vectors.npz for verification")

Resource          Used   Available     Util
--------------------------------------------
LUT              24154       53200    45.4%
FF               26992      106400    25.4%
BRAM_36K             3         140     2.1%
DSP                 30         220    13.6%

DEPLOYMENT PACKAGE: $HOME/project/tiny_output/deploy
    bitfile
    bitfile/finn-accel.bit
    bitfile/finn-accel.hwh
    driver
    driver/driver.py
    driver/driver_base.py
    driver/finn
    driver/finn/util
    driver/finn/util/__init__.py
    driver/finn/util/data_packing.py
    driver/qonnx
    driver/qonnx/core

Copy to the board:
  scp -r $HOME/project/tiny_output/deploy xilinx@<board-ip>:~/tiny_astar
  also copy tiny_test_vectors.npz for verification


## 8. Tomorrow, on the board

```bash
scp -r ~/fpga_hw2/tiny_output/deploy xilinx@<board-ip>:~/tiny_astar
scp ~/fpga_hw2/tiny_test_vectors.npz xilinx@<board-ip>:~/tiny_astar/
```

Then on the PYNQ-Z2, as in HW2's `03_board_inference`:

```python
from driver import io_shape_dict
from driver_base import FINNExampleOverlay
import numpy as np, time

acc = FINNExampleOverlay(bitfile_name="resizer.bit", platform="zynq-iodma",
                         io_shape_dict=io_shape_dict, batch_size=1)

d = np.load("tiny_test_vectors.npz")
out = acc.execute(d["x"][:1])
print("max error vs model:", np.abs(out - d["y"][:1]).max())

t0 = time.time()
for i in range(100):
    acc.execute(d["x"][i % 64:i % 64 + 1])
print(f"{(time.time()-t0)/100*1000:.2f} ms/frame on hardware")
```

Two numbers come out of that: **correctness** (hardware matches the model) and
**measured latency** (not an estimate). Against the 6.70 ms CPU baseline, that
is the first real end-to-end result.

### Then grow it

With a working pipeline, each step is one variable:

1. `MAC_BUDGET` up — more parallelism, faster, more LUTs.
2. `CHANNELS` wider — better `p_opt`/`p_exp`, more resources.
3. Repeat and plot.

Every point is a `(accuracy, resources, throughput)` triple. Three or four of
them give the frontier — including the measured ceiling where the original
391k encoder needed 3.47× the device. That is a stronger result than one
bitfile, and it is reached by growing a working system rather than shrinking a
broken one.